# Vesselbound / Rift Realm — Batch 1 Multi-View 3D Shape Generator (Colab Pro GPU)

Runs **Hunyuan3D-2mv** (`tencent/Hunyuan3D-2mv`) on Colab Pro GPU (L4 / A100) using orthographic multi-view turnaround cutouts (`front`, `left`, `back`) for:
- **Seraph-9** (`seraph_9` — body-only front + back; wings/halo/lance mounted as articulated parts locally)
- **Lumen** (`lumen` — front + left + back)
- **Sylva** (`sylva` — front + left + back)
- **Pyre** (`pyre` — front + left + back)
- **Nyx** (`nyx` — front + left + back)

**Make sure Runtime → Change runtime type is set to `L4 GPU` or `A100 GPU` before running!**

In [ ]:
# Cell 1: Verify GPU runtime and install Hunyuan3D-2 (pure-Python/wheel shape-gen dependencies, no slow CUDA compile)
import os, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU attached! Go to Runtime -> Change runtime type -> L4 GPU or A100 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)')

if not os.path.isdir('/content/Hunyuan3D-2'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent/Hunyuan3D-2.git', '/content/Hunyuan3D-2'], check=True)

pkgs = [
    'trimesh', 'einops', 'omegaconf', 'safetensors', 'huggingface_hub',
    'scikit-image', 'pymeshlab', 'opencv-python-headless', 'scipy',
    'pygltflib', 'accelerate', 'diffusers', 'transformers', 'timm'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

import hy3dgen
print('Hunyuan3D-2 import OK:', hy3dgen.__file__)

In [ ]:
# Cell 2: Unpack /content/batch1_inputs.zip (upload via Colab Files sidebar or upload prompt)
import json, os, zipfile

ZIP_PATH = '/content/batch1_inputs.zip'
if not os.path.isfile(ZIP_PATH):
    from google.colab import files
    print('Please upload batch1_inputs.zip...')
    uploaded = files.upload()
    first_name = next(iter(uploaded.keys()))
    if first_name != 'batch1_inputs.zip':
        os.rename(first_name, ZIP_PATH)

IN_DIR = '/content/batch1_inputs'
os.makedirs(IN_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(IN_DIR)

with open(os.path.join(IN_DIR, 'manifest.json'), 'r', encoding='utf-8') as f:
    MANIFEST = json.load(f)

print(f"Loaded batch: {MANIFEST['batch']} ({len(MANIFEST['characters'])} characters)")
for item in MANIFEST['characters']:
    print(f"  - {item['id']} ({item['rig_type']}): views={list(item['mv_views'].keys())}, octree={item['octree_resolution']}, steps={item['steps']}")

In [ ]:
# Cell 3: Run multi-view Hunyuan3D-2mv across all Batch 1 characters
import gc, json, os, sys, time
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt
from PIL import Image

if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

from hy3dgen.shapegen import (
    Hunyuan3DDiTFlowMatchingPipeline,
    FloaterRemover,
    DegenerateFaceRemover,
    FaceReducer,
)

OUT_DIR = '/content/batch1_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
use_turbo = vram_gb < 18.0
subfolder = 'hunyuan3d-dit-v2-mv-turbo' if use_turbo else 'hunyuan3d-dit-v2-mv'
print(f'Loading tencent/Hunyuan3D-2mv ({subfolder}, fp16) on {vram_gb:.1f} GB VRAM...')
t_load = time.time()
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mv',
    subfolder=subfolder,
    variant='fp16',
)
if use_turbo:
    pipeline.enable_flashvdm(mc_algo='mc')
print(f'Model loaded in {time.time() - t_load:.1f} s')

floater_remover = FloaterRemover()
degen_remover = DegenerateFaceRemover()
face_reducer = FaceReducer()

def normalize_mesh(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)
    if mesh.is_volume and mesh.volume < 0:
        mesh.invert()
    bounds = mesh.bounds
    cx = 0.5 * (bounds[0][0] + bounds[1][0])
    cz = 0.5 * (bounds[0][2] + bounds[1][2])
    miny = bounds[0][1]
    mesh.apply_translation([-cx, -miny, -cz])
    return mesh

report = {'gpu': torch.cuda.get_device_name(0), 'subfolder': subfolder, 'results': []}
meshes_for_preview = []

for item in MANIFEST['characters']:
    cid = item['id']
    c_out = os.path.join(OUT_DIR, cid)
    os.makedirs(c_out, exist_ok=True)
    glb_path = os.path.join(c_out, 'shape_raw.glb')

    if os.path.isfile(glb_path):
        print(f'[{cid}] Already generated, loading existing GLB...')
        mesh = trimesh.load(glb_path, force='mesh')
        meshes_for_preview.append((cid, mesh))
        report['results'].append({'id': cid, 'cached': True, 'faces': int(len(mesh.faces))})
        continue

    views = {}
    for vkey, fname in item['mv_views'].items():
        vpath = os.path.join('/content/batch1_inputs', cid, fname)
        views[vkey] = Image.open(vpath).convert('RGBA')

    steps = 8 if use_turbo else int(item.get('steps', 30))
    octree = int(item.get('octree_resolution', 384))
    face_limit = int(item.get('face_limit', 200000))
    seed = int(item.get('seed', 42))

    print(f'[{cid}] Generating multi-view ({list(views.keys())}), steps={steps}, octree={octree}...')
    t0 = time.time()
    mesh = pipeline(
        image=views,
        num_inference_steps=steps,
        octree_resolution=octree,
        num_chunks=20000,
        mc_algo='mc',
        generator=torch.manual_seed(seed),
        output_type='trimesh',
    )[0]
    raw_faces = int(len(mesh.faces))
    mesh = floater_remover(mesh)
    mesh = degen_remover(mesh)
    mesh = face_reducer(mesh, max_facenum=face_limit)
    mesh = normalize_mesh(mesh)
    mesh.export(glb_path)
    dt = time.time() - t0
    mb = os.path.getsize(glb_path) / 1e6
    print(f'[{cid}] Done in {dt:.1f}s -> {len(mesh.faces)} faces (raw {raw_faces}), {mb:.2f} MB, extents={np.round(mesh.extents, 3).tolist()}')
    report['results'].append({
        'id': cid, 'seconds': round(dt, 1), 'raw_faces': raw_faces,
        'faces': int(len(mesh.faces)), 'mb': round(mb, 2),
        'extents': np.round(mesh.extents, 4).tolist(),
    })
    meshes_for_preview.append((cid, mesh))
    gc.collect()
    torch.cuda.empty_cache()

with open(os.path.join(OUT_DIR, 'report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

# Render fast orthographic point-silhouette preview (front / left / back) for immediate visual QA
fig, axes = plt.subplots(len(meshes_for_preview), 3, figsize=(9, 3 * len(meshes_for_preview)), facecolor='#11131a')
if len(meshes_for_preview) == 1:
    axes = [axes]
for row, (cid, mesh) in enumerate(meshes_for_preview):
    v = np.asarray(mesh.vertices)
    step = max(1, len(v) // 18000)
    vs = v[::step]
    # front (+Z): x vs y ; left (+X): -z vs y ; back (-Z): -x vs y
    projs = [('front', vs[:, 0], vs[:, 1], vs[:, 2]), ('left', -vs[:, 2], vs[:, 1], vs[:, 0]), ('back', -vs[:, 0], vs[:, 1], -vs[:, 2])]
    for col, (lbl, px, py, depth) in enumerate(projs):
        ax = axes[row][col]
        ax.set_facecolor('#11131a')
        order = np.argsort(depth)
        ax.scatter(px[order], py[order], c=depth[order], cmap='viridis', s=0.6, linewidths=0)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{cid} ({lbl})', color='white', fontsize=10)
plt.tight_layout()
prev_path = os.path.join(OUT_DIR, 'batch1_shapes_preview.png')
plt.savefig(prev_path, dpi=140, facecolor='#11131a')
plt.show()
print('Preview saved:', prev_path)

In [ ]:
# Cell 4: Zip all generated GLBs + report + preview and trigger download
import os, zipfile
from google.colab import files

OUT_DIR = '/content/batch1_outputs'
ZIP_OUT = '/content/batch1_shapes.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT_DIR):
        for fn in sorted(fnames):
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, OUT_DIR)
            zf.write(full, rel)

print(f'READY: {ZIP_OUT} ({os.path.getsize(ZIP_OUT) / 1e6:.2f} MB)')
files.download(ZIP_OUT)